# nb00: Stage 0 Diagnostics on nb03_best.pt

* * *

**QUESTION:** Does the trained JEPA system (frozen random backbone + trained projection/predictor) capture genuine temporal structure, or is val_jepa=0.064 a statistical artifact of the EMA-paired projections becoming similar regardless of temporal position?

**H1:** The trained predictor maps z_price to a z_pred that is more cosine-similar to the true future z_target than to a randomly permuted z_target. The representation is temporal.

**H0 (null):** Shuffled and true z_target cosine similarities are within 20% of each other. The low val_jepa reflects EMA collapse (projection heads converged to the same mapping regardless of input window), not temporal prediction.

**MEASURED TARGET:**
- Primary: ratio = shuffled_cosine_mean / true_cosine_mean, computed over 50 val batches (batch_size=256, ~12,800 samples), seed=42
- Supporting: untrained_jepa_mean (fresh random init, same arch), adjacent_cosine_mean (last context step vs first target step, 6-d), effective_rank of val z_price covariance, mean_r2 of z_price vs volatility proxy
- All metrics: mean and std reported. Val split = last 15% of each ticker chronologically.

**DECISION RULE:**
- ratio >= 0.8: degenerate. Do NOT proceed to more training. Investigate frozen pretrained backbone (Moirai/TimesFM in a clean env) or reconstruction pretraining before JEPA.
- ratio < 0.8: temporal signal present. Proceed to Stage 1 linear probe to quantify downstream value.
- Additionally: effective_rank < 10 means severe collapse regardless of shuffle ratio. Address before Stage 1.

**PRIORS / ASSUMPTIONS:**
- Inputs are normalized log-returns (per-sample normalization, causal: context mean/std applied to both windows). Raw price level is not present.
- Backbone is a frozen random transformer (TransformerPriceEncoder, 6 layers, d_model=512). Only projection head and predictor were trained.
- EMA decay=0.999: target encoder tracks context encoder closely. Hard-copying price_encoder into target_encoder in load_checkpoint is a reasonable approximation.
- The best checkpoint (val_jepa=0.064) is from nb03/nb03b. If the checkpoint config differs from the default JEPAConfig, set it in the config cell below.
- Known confound: VICReg variance hinge acts on train batches only. Val z_std expected ~0.63-0.74, lower than train ~1.07.

**FALSIFIER:** If shuffled cosine similarity equals or exceeds true cosine similarity (ratio >= 0.8), H1 is falsified. Equally: if effective_rank of val z_price < 5, the representation is too collapsed to carry structure.

**RESULT:** *(fill in after running)*

**DECISION + NEXT ACTION:** *(fill in after running)*

* * *

In [ ]:
# == Colab setup (skipped in VS Code / local kernel) ==
import os, sys

IN_COLAB = 'google.colab' in sys.modules
IN_VSCODE = 'VSCODE_PID' in os.environ or 'VSCODE_CWD' in os.environ
if IN_VSCODE:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content
    !git clone https://github.com/shreyasnat2804/JEPA-quant.git 2>/dev/null || (cd JEPA-quant && git pull)
    %cd /content/JEPA-quant
    %pip install -q einops matplotlib pyarrow

# Add src to path regardless of environment
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..' if 'notebooks' in os.getcwd() else '.'))
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('repo_root:', repo_root)
print('IN_COLAB:', IN_COLAB, '  IN_VSCODE:', IN_VSCODE)

In [ ]:
# == autoreload shim (imp removed in Python 3.12) ==
import types, importlib
if 'imp' not in sys.modules:
    _imp_shim = types.ModuleType('imp')
    _imp_shim.reload = importlib.reload
    sys.modules['imp'] = _imp_shim
%load_ext autoreload
%autoreload 2

In [ ]:
# == user configuration: edit before running ==
import os

# Path to the best checkpoint saved by JEPATrainer
# (keys: step, val_jepa, price_encoder, predictor, opt)
# Colab with Drive: '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/nb03_best.pt'
# Local: absolute path on your machine
CHECKPOINT_PATH = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/nb03_best.pt'

# Directory containing per-ticker .parquet files (same as used for training)
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/raw/stocks'

# Number of val batches per diagnostic. 50 * 256 = 12,800 samples. Do not go below 30.
N_BATCHES = 50

# Fixed seed for all randomness in diagnostics
SEED = 42

# Device: 'cuda' if available, else 'cpu'. Diagnostics run fine on CPU.
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# == build config and load checkpoint ==
# The config must match the architecture used to produce the checkpoint.
# Default JEPAConfig: transformer backend, freeze_backbone=True, d_model=512, n_layers=6, latent_dim=256.
# Override here if nb03 used different hyperparameters.
import torch
from jepa_quant.config import (
    JEPAConfig, PriceEncoderConfig, DataConfig, TrainConfig,
    PredictorConfig, EMAConfig, VICRegConfig,
)
from jepa_quant.eval.diagnostics import load_checkpoint
from jepa_quant.data.price_dataset import PriceWindowDataset
from torch.utils.data import DataLoader

cfg = JEPAConfig(
    price_encoder=PriceEncoderConfig(
        backend='transformer',
        n_features=6,
        context_length=64,
        d_model=512,
        n_heads=8,
        n_layers=6,
        latent_dim=256,
        freeze_backbone=True,
    ),
    data=DataConfig(
        data_dir=DATA_DIR,
        context_length=64,
        horizon=16,
        val_fraction=0.15,
        normalize=True,
    ),
    train=TrainConfig(
        batch_size=256,
        num_workers=2,
        device=DEVICE,
        seed=SEED,
    ),
)

# Val loader for tests 1-3 and 5 (test 4 builds its own unnormalized loader)
val_ds = PriceWindowDataset(cfg.data, 'val')
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.train.batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=cfg.train.num_workers,
)
print(f'val windows: {len(val_ds)}')

components = load_checkpoint(CHECKPOINT_PATH, cfg, device=DEVICE)
print(f'checkpoint loaded: {CHECKPOINT_PATH}')

## Test 1: Shuffled-Target Control (decisive)

Computes true and randomly-paired cosine similarity. Decision gate: ratio >= 0.8 means degenerate.

In [ ]:
from jepa_quant.eval.diagnostics import shuffled_target_control

ctrl = shuffled_target_control(components, val_loader, n_batches=N_BATCHES, seed=SEED, device=DEVICE)

print('=== Shuffled-Target Control ===')
print(f"  true  cosine:    {ctrl['true_cosine_mean']:.4f} +/- {ctrl['true_cosine_std']:.4f}  (jepa={ctrl['true_jepa']:.4f})")
print(f"  shuf  cosine:    {ctrl['shuffled_cosine_mean']:.4f} +/- {ctrl['shuffled_cosine_std']:.4f}  (jepa={ctrl['shuffled_jepa']:.4f})")
print(f"  ratio shuf/true: {ctrl['ratio_shuffled_over_true']:.4f}")
print(f"  n_samples:       {ctrl['n_samples']}")
print(f"  VERDICT:         {ctrl['verdict'].upper()}")
print()
if ctrl['verdict'] == 'degenerate':
    print('  ratio >= 0.8: representations are degenerate. Fix backbone before any further training.')
else:
    print('  ratio < 0.8: genuine temporal signal. Proceed to Stage 1 linear probe.')

## Test 2: Baselines

(a) Untrained val_jepa: same architecture, no training. Should be larger than trained val_jepa.  
(b) Adjacent-timestep cosine in 6-d log-return space: raw-feature correlation context.

In [ ]:
from jepa_quant.eval.diagnostics import compute_baselines

bl = compute_baselines(cfg, val_loader, n_batches=N_BATCHES, seed=SEED, device=DEVICE)

print('=== Baselines ===')
print(f"  untrained jepa:       {bl['untrained_jepa_mean']:.4f} +/- {bl['untrained_jepa_std']:.4f}")
print(f"  trained val_jepa:     {ctrl['true_jepa']:.4f}  (for comparison)")
print(f"  adjacent cosine (6d): {bl['adjacent_cosine_mean']:.4f} +/- {bl['adjacent_cosine_std']:.4f}")
print(f"  n_batches:            {bl['n_batches']}")
print()
trained_better = ctrl['true_jepa'] < bl['untrained_jepa_mean']
print(f"  trained < untrained:  {trained_better}  (True = training improved over random init)")
if not trained_better:
    print('  WARNING: trained model is NOT better than random init. Check for optimizer collapse.')

## Test 3: Collapse Audit

Per-dim std, eigenspectrum, effective rank. Effective rank near 256 = full utilization. Near 1-10 = severe collapse.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from jepa_quant.eval.diagnostics import collapse_audit

audit = collapse_audit(components, val_loader, n_batches=N_BATCHES, seed=SEED, device=DEVICE)

print('=== Collapse Audit ===')
print(f"  z_std mean:          {audit['z_std_mean']:.4f}  (train ~1.07, val ~0.63-0.74 from training logs)")
print(f"  z_std range:         [{audit['z_std_min']:.4f}, {audit['z_std_max']:.4f}]")
print(f"  effective rank:      {audit['effective_rank']:.1f} / {audit['n_dims']}")
print(f"  top-1 eig share:     {audit['top1_eigenvalue_share']:.3f}  (>0.5 = one direction dominates)")
print(f"  top-10 eig share:    {audit['top10_eigenvalue_share']:.3f}  (<0.5 = broad utilization)")
print(f"  n_samples:           {audit['n_samples']}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

std_sorted = sorted(audit['per_dim_std'], reverse=True)
axes[0].bar(range(len(std_sorted)), std_sorted, color='steelblue', alpha=0.7)
axes[0].axhline(1.0, color='red', linestyle='dashed', label='VICReg target (gamma=1)')
axes[0].set_xlabel('dimension (sorted by std)')
axes[0].set_ylabel('std')
axes[0].set_title(f'Per-dim z_price std (val)  mean={audit["z_std_mean"]:.3f}')
axes[0].legend()

eigs = np.array(audit['eigenvalues'])
axes[1].semilogy(eigs, color='darkorange', alpha=0.8)
axes[1].set_xlabel('component')
axes[1].set_ylabel('eigenvalue (log scale)')
axes[1].set_title(f'Covariance eigenspectrum  eff_rank={audit["effective_rank"]:.1f}')

plt.tight_layout()
plt.show()

## Test 4: Level-Dependence Test

R^2 of each z_price dimension regressed on the per-window volatility proxy (mean temporal std of unnormalized log-returns). High mean R^2 means the representation mostly encodes scale that normalization was supposed to remove.

In [ ]:
from jepa_quant.eval.diagnostics import level_dependence_test

lvl = level_dependence_test(components, cfg, n_batches=N_BATCHES, seed=SEED, device=DEVICE)

print('=== Level-Dependence Test ===')
print(f"  mean R^2 (vol -> z): {lvl['mean_r2']:.4f}")
print(f"  median R^2:          {lvl['median_r2']:.4f}")
print(f"  max R^2:             {lvl['max_r2']:.4f}")
print(f"  dims with R^2>0.1:   {lvl['n_dims_r2_gt_0p1']} / {len(lvl['r2_per_dim'])}")
print(f"  dims with R^2>0.3:   {lvl['n_dims_r2_gt_0p3']} / {len(lvl['r2_per_dim'])}")
print(f"  top-10 R^2:          {[round(v,3) for v in lvl['top10_r2']]}")
print()
if lvl['mean_r2'] > 0.3:
    print('  mean R^2 > 0.3: representation strongly encodes volatility regime.')
    print('  If combined with shuffled verdict=degenerate: encoder encodes scale, not temporal structure.')
else:
    print('  mean R^2 <= 0.3: representation is not dominated by the volatility proxy.')

fig, ax = plt.subplots(figsize=(10, 3))
r2_sorted = sorted(lvl['r2_per_dim'], reverse=True)
ax.bar(range(len(r2_sorted)), r2_sorted, color='mediumseagreen', alpha=0.7)
ax.axhline(0.1, color='orange', linestyle='dashed', label='R^2=0.1')
ax.axhline(0.3, color='red', linestyle='dashed', label='R^2=0.3')
ax.set_xlabel('dimension (sorted by R^2)')
ax.set_ylabel('R^2 (vol proxy vs z_price)')
ax.set_title(f'Volatility dependence per z_price dim  mean={lvl["mean_r2"]:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

## Test 5: Regime Clustering (confirmatory)

PCA of val z_price colored by volatility quintile and return sign. Use this to support or question tests 1-3, not to make a decision.

In [ ]:
from jepa_quant.eval.diagnostics import regime_clustering

pca = regime_clustering(components, val_loader, n_batches=N_BATCHES, seed=SEED, device=DEVICE)

print('=== Regime Clustering (PCA) ===')
print(f"  effective rank:       {pca['effective_rank']:.1f}")
print(f"  top-2 PC variance:    {pca['top2_pc_variance']:.3f}")
print(f"  top-10 PC variance:   {pca['top10_pc_variance']:.3f}")
print(f"  variance ratios[:5]:  {[round(v,4) for v in pca['pca_variance_ratios_top20'][:5]]}")
print(f"  n_samples:            {pca['n_samples']}")

proj = np.array(pca['pca2_projections'])
vol_q = np.array(pca['vol_quintile_labels'])
ret_s = np.array(pca['return_sign_labels'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc0 = axes[0].scatter(proj[:,0], proj[:,1], c=vol_q, cmap='RdYlGn', s=5, alpha=0.5)
plt.colorbar(sc0, ax=axes[0], label='vol quintile (0=low, 4=high)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('z_price PCA: volatility quintile')

colors = ['#ef4444' if s == 0 else '#22c55e' for s in ret_s]
axes[1].scatter(proj[:,0], proj[:,1], c=colors, s=5, alpha=0.5)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('z_price PCA: return sign (green=positive, red=negative)')

plt.tight_layout()
plt.show()

## Summary and Decision

Fill in the RESULT and DECISION fields in the header cell after running all tests above.

* * *

### Quick reference: decision tree

```
ratio < 0.8 AND effective_rank >= 20
  -> temporal signal + non-collapsed. Run Stage 1 linear probe.

ratio < 0.8 AND effective_rank < 20
  -> temporal signal but collapsed. Fix collapse first (increase lambda_v, more data).
     Then Stage 1.

ratio >= 0.8 AND mean_r2 > 0.3
  -> degenerate + vol-dominated. Encodes scale but not temporal structure.
     Investigate frozen pretrained backbone or reconstruction pretraining.

ratio >= 0.8 AND mean_r2 <= 0.3
  -> degenerate + not vol-dominated. Pure EMA collapse. Backbone too random.
     Same fix: pretrained backbone.
```

In [ ]:
# == consolidated results dict for log entry ==
import json

results_summary = {
    'checkpoint': CHECKPOINT_PATH,
    'n_batches': N_BATCHES,
    'seed': SEED,
    'shuffled_control': {
        'true_cosine_mean': ctrl['true_cosine_mean'],
        'shuffled_cosine_mean': ctrl['shuffled_cosine_mean'],
        'ratio': ctrl['ratio_shuffled_over_true'],
        'verdict': ctrl['verdict'],
        'n_samples': ctrl['n_samples'],
    },
    'baselines': {
        'untrained_jepa_mean': bl['untrained_jepa_mean'],
        'trained_jepa': ctrl['true_jepa'],
        'adjacent_cosine_mean': bl['adjacent_cosine_mean'],
    },
    'collapse': {
        'z_std_mean': audit['z_std_mean'],
        'effective_rank': audit['effective_rank'],
        'top10_eig_share': audit['top10_eigenvalue_share'],
    },
    'level_dependence': {
        'mean_r2': lvl['mean_r2'],
        'n_dims_r2_gt_0p3': lvl['n_dims_r2_gt_0p3'],
    },
    'pca': {
        'effective_rank': pca['effective_rank'],
        'top2_pc_variance': pca['top2_pc_variance'],
    },
}

print(json.dumps(results_summary, indent=2))